# 🔬 Notebook 3: Gmail — Deep Dive: SMTP, Search, Storage tiers

## 🛠️ Setup

```bash
cd 06-system-designs/gmail
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Deep dive 1 — SMTP handshake

When our outbound server sends mail to `alice@example.com`:

```
1. DNS query:    MX records for example.com  →  mx1.example.com
2. TCP connect to mx1.example.com:25
3. SMTP conversation:
   S: 220 mx1.example.com ESMTP ready
   C: EHLO our.server.com
   S: 250-mx1.example.com ... 250 OK
   C: MAIL FROM:<us@ourdomain.com>
   S: 250 OK
   C: RCPT TO:<alice@example.com>
   S: 250 OK
   C: DATA
   S: 354 Start mail input
   C: Subject: Hi\r\nFrom: ...\r\n\r\nHello!\r\n.\r\n
   S: 250 Queued as ABC123
   C: QUIT
```

`MAIL FROM` is the "envelope sender" (for bounces). `RCPT TO` is the envelope recipient.
Multiple `RCPT TO` per message → server fan-out to multiple users on its side.


In [ ]:
# Simulate a tiny SMTP-like state machine so the sequence clicks.
class MiniSmtp:
    def __init__(self):
        self.state = "INIT"; self.mail_from = None; self.rcpts = []; self.data = None
    def handle(self, line: str) -> str:
        l = line.upper()
        if self.state == "INIT" and l.startswith("EHLO"):
            self.state = "GREETED"; return "250 OK"
        if self.state == "GREETED" and l.startswith("MAIL FROM:"):
            self.mail_from = line.split(":",1)[1].strip()
            self.state = "GOT_SENDER"; return "250 OK"
        if self.state in ("GOT_SENDER","GOT_RCPT") and l.startswith("RCPT TO:"):
            self.rcpts.append(line.split(":",1)[1].strip()); self.state = "GOT_RCPT"
            return "250 OK"
        if self.state == "GOT_RCPT" and l == "DATA":
            self.state = "DATA"; return "354 start mail input"
        if self.state == "DATA":
            self.data = line; self.state = "DONE"; return "250 Queued"
        return "500 Bad sequence"

s = MiniSmtp()
for cmd in ["EHLO me.com", "MAIL FROM:<a@me.com>", "RCPT TO:<b@you.com>",
            "RCPT TO:<c@you.com>", "DATA", "Hello!"]:
    print(f"C> {cmd}")
    print(f"S> {s.handle(cmd)}")
print("envelope:", s.mail_from, "→", s.rcpts)


## Deep dive 2 — full-text search

Search must handle `from:alice subject:report attachment:yes after:2025/01/01 quarterly`.

Approach: **per-user inverted index** (Lucene/Elasticsearch), with structured fields:
- Text fields: `subject`, `body`.
- Keyword fields: `from`, `to`, `label`.
- Date: `received_at`.
- Boolean flags: `has_attachment`, `has_link`.

Each user has their own shard/tenant. Indexing is async: a worker consumes new-message
events and writes to the index.

### Why per-user?
- Privacy: one user's search should never touch another's data.
- Size: most users have <50k messages — a tiny index; aggressive caching works.


In [ ]:
# Tiny email search implementation with structured fields + tokens
import re
from collections import defaultdict

class MiniSearch:
    def __init__(self):
        self.messages: list[dict] = []
        self.term_idx: dict[str, set[int]] = defaultdict(set)

    def add(self, msg: dict):
        mid = len(self.messages); self.messages.append(msg)
        for w in re.findall(r"\w+", (msg["subject"] + " " + msg["body"]).lower()):
            self.term_idx[w].add(mid)

    def search(self, query: str) -> list[dict]:
        # very simple parser: from:X, has:attachment, or bare words
        tokens = query.lower().split()
        candidates = None
        filters = []
        for t in tokens:
            if ":" in t:
                k, v = t.split(":", 1)
                filters.append((k, v))
            else:
                s = self.term_idx.get(t, set())
                candidates = s if candidates is None else candidates & s
        results = list(candidates) if candidates is not None else list(range(len(self.messages)))
        for k, v in filters:
            if k == "from":
                results = [i for i in results if v in self.messages[i]["from"].lower()]
            elif k == "has" and v == "attachment":
                results = [i for i in results if self.messages[i].get("has_attachment")]
        return [self.messages[i] for i in results]

s = MiniSearch()
s.add({"from":"alice@ex.com","subject":"Q3 report","body":"see attached","has_attachment":True})
s.add({"from":"bob@ex.com","subject":"Lunch","body":"q3 discussion"})
s.add({"from":"alice@ex.com","subject":"Random","body":"hello"})

print("from:alice →", [m["subject"] for m in s.search("from:alice")])
print("q3 →", [m["subject"] for m in s.search("q3")])
print("q3 has:attachment →", [m["subject"] for m in s.search("q3 has:attachment")])


## Deep dive 3 — storage tiers

A user's inbox has a **power law**: 1% of messages are read 90% of the time (recent).

```
   hot tier:  SSD / in-memory cache  (last 30 days)
   warm tier: HDD / standard object store (30 days – 1 year)
   cold tier: archive storage (Glacier-class) (> 1 year)
```

A nightly migration job moves messages between tiers. Reads from cold tiers are slow but
acceptable for ancient searches. This cuts storage cost ~10× at large scale.
